# Create SU2C Awards (Stand Up To Cancer)

Creates Stand Up To Cancer research awards from SU2C's own Scientific Summit
Program Guides, linked from https://standuptocancer.org/science/portfolio-overview/
(the 2024 guide "includes the entire SU2C science portfolio since the start of
the organization"; the 2025 and 2026 guides add the newest teams and awards).
Pathways: Dream Teams, Research Teams, Convergence teams, Catalyst teams and the
Health Equity Breakthrough Team (one row per SU2C grant number), Innovative
Research Grants (classes 2009, 2011, 2016, 2017), Phillip A. Sharp Innovation in
Collaboration Awards (2014-2024), named awards (Ziskin Prize, Jim Toth Sr.,
Peggy Prescott, Golden Arrow, Sharp Tank, Sharp Challenge, Maverick, Cless
Family, Nina Nicolai) and the 43 SU2C Kimmel Scholars (2009-2011).
**248 rows: 102 team grant numbers (2009-2026; 97 SU2C, 5 SU2C Canada), 46 Innovative Research Grants, 38 Sharp awards, 19 named awards, 43 Kimmel Scholars; 100% lead name, 100% title or scheme, team dates 100%, amounts only for the Ziskin Prize** (local run 2026-10-01).

AACR administers most SU2C grants as SU2C's scientific partner, but the money is
SU2C's; the AACR ingest (priority 560) excludes all SU2C pages, so there is no
overlap. Partner co-funded teams (SU2C-Lustgarten, SU2C-CRUK, SU2C-Dutch Cancer
Society, Van Andel Institute-SU2C, Pancreatic Cancer Collective, Torrey Coast
...) stay under SU2C with the partner visible in `funder_scheme` (the team name).

**Routing (§2.3.2):** teams named "SU2C Canada ..." (5 grant numbers) are funded
by Stand Up To Cancer Canada, which has its own OpenAlex funder row
(F4320315605); the script sets `route_funder_id` and this notebook joins the
funder struct on it. Everything else is SU2C (F4320332898).

**Prerequisites:**
- Run `scripts/local/su2c_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/su2c/su2c_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** teams ship SU2C's grant number as printed in the
guide (`DT5918`, `RT6186`, `CV6204` ...). Citing works cite the AACR-era form
for pre-2019 teams (`SU2C-AACR-DT0209`, `SU2C-AACR-IRG 20-17`) or `SU2C#6143`,
`RT6187` for newer ones; the guides do not print the AACR-era numbers, so only
exact matches collapse. Grants without a printed number get a synthetic key
`SU2C-{IRG|IRG-SELIG|SHARP|SHARP-ZISKIN|ZISKIN|TOTH|...|KIMMEL}-{year}-{lead-slug}`.

**Dates:** teams carry "Grant Term: Month YYYY - Month YYYY" (start = first day,
end = last day of the month); individual grants and awards carry the class/award
year only (start_year); Kimmel Scholars have no per-person year (2009-2011).

**Amounts:** not published per grant; only the Ziskin Prize states its amount
("a one-year, $250,000 grant ... shared by two scientists"), shipped as USD 250,000.

**Twins:** Entertainment Industry Foundation (F4320309149) has the same ROR
(01dmpb598; SU2C was an EIF program until it became its own 501(c)(3)) and
carries 15 SU2C-AACR-* citation stubs. SU2C Canada (F4320315605) is a separate
entity (rows routed to it above).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320332898` (and `4320315605` for SU2C Canada rows)
- ror_id / doi: from `openalex.funders.funders`

**Priority:** `596` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.su2c_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/su2c/su2c_projects.parquet`;

In [ ]:
%sql
SELECT award_kind, route_funder_id, COUNT(*) as n FROM openalex.awards.su2c_raw GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
DESCRIBE openalex.awards.su2c_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.su2c_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
Both funder ids are Crossref-registered (F4320*), so each MUST resolve to exactly
1 row in `openalex.funders.funders`. If not, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id IN (4320332898, 4320315605);

## Step 2: Create SU2C Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.su2c_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id IN (4320332898, 4320315605)  -- SU2C, SU2C Canada
),

base AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        g.*,
        TRY_CAST(g.route_funder_id AS BIGINT) AS resolved_funder_id,
        CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(g.lead_given_name), ''),
            'family_name', NULLIF(TRIM(g.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.lead_institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead_struct,
        CASE WHEN NULLIF(TRIM(g.co_lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(g.co_lead_given_name), ''),
            'family_name', NULLIF(TRIM(g.co_lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.co_lead_institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS co_lead_struct,
        transform(
            from_json(g.investigators_json, 'array<struct<role:string,given:string,family:string,inst:string>>'),
            x -> named_struct(
                'given_name', NULLIF(TRIM(x.given), ''),
                'family_name', NULLIF(TRIM(x.family), ''),
                'orcid', CAST(NULL AS STRING),
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(x.inst), ''),
                    'country', CAST(NULL AS STRING),
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                )
            )
        ) AS investigators_arr,
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') AS sd,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') AS ed
    FROM openalex.awards.su2c_raw g
    WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    COALESCE(
        NULLIF(TRIM(g.title), ''),
        NULLIF(TRIM(g.team_name), ''),
        CONCAT(g.funder_scheme, COALESCE(CONCAT(' ', g.start_year), ''), ': ',
               CONCAT_WS(' ', g.lead_given_name, g.lead_family_name))
    ) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.award_kind = 'kimmel' THEN 'fellowship' ELSE 'research' END as funding_type,
    NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,
    'su2c_program_guides' as provenance,
    g.sd as start_date,
    g.ed as end_date,
    COALESCE(YEAR(g.sd), TRY_CAST(g.start_year AS INT)) as start_year,
    YEAR(g.ed) as end_year,
    g.lead_struct as lead_investigator,
    g.co_lead_struct as co_lead_investigator,
    CASE
        WHEN size(g.investigators_arr) > 0 THEN g.investigators_arr
        WHEN g.lead_struct IS NOT NULL THEN ARRAY(g.lead_struct)
    END as investigators,
    g.source_pdf as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM base g
JOIN src_funder f ON f.funder_id = g.resolved_funder_id;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'su2c_program_guides' AND priority = 596;

-- Insert into openalex_awards_raw with priority.
-- Priority 596: direct-from-funder ingest of SU2C's own program guides.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    596 as priority
FROM openalex.awards.su2c_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids,
       (SELECT COUNT(*) FROM openalex.awards.su2c_raw) as raw_rows
FROM openalex.awards.su2c_awards;

In [ ]:
%sql
-- §2.1.1 shape check: printed SU2C grant numbers or the synthetic SU2C-* form.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^(DT|RT|CT|CV|BT)[0-9]{4}$' THEN 1 ELSE 0 END) as grant_numbers,
    SUM(CASE WHEN funder_award_id RLIKE '^SU2C-[A-Z-]+-([0-9]{4}-)?[a-z0-9-]+$' THEN 1 ELSE 0 END) as synthetic,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^(DT|RT|CT|CV|BT)[0-9]{4}$'
              AND funder_award_id NOT RLIKE '^SU2C-[A-Z-]+-([0-9]{4}-)?[a-z0-9-]+$' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.su2c_awards;

In [ ]:
%sql
SELECT funder_id, funder.display_name, funder_scheme, COUNT(*) as n
FROM openalex.awards.su2c_awards
GROUP BY 1, 2, 3 ORDER BY 1, n DESC;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_date, end_date, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, size(investigators) as n_inv
FROM openalex.awards.su2c_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.su2c_awards GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.su2c_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.su2c_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage (amounts are not published per grant: only the Ziskin Prize).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    COUNT(start_date) as has_start_date,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.su2c_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, funder_id, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'su2c_program_guides'
GROUP BY provenance, priority, funder_id;